In [1]:
print("Hello, World!")

Hello, World!


In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.metrics.pairwise import cosine_similarity

ImportError: DLL load failed while importing _bounded_integers: An Application Control policy has blocked this file.

In [ ]:


# 1. Load Metadata and Filter Dataset
# Assumes styles.csv and images folder are in the current directory
df = pd.read_csv('styles.csv', error_bad_lines=False)
df['image_path'] = df['id'].apply(lambda x: f"images/{x}.jpg")

# Select 10 distinct categories and 5 images from each
categories = df['masterCategory'].unique()[:10]
subset_df = pd.concat([df[df['masterCategory'] == cat].head(5) for cat in categories])

# 2. Initialize ResNet50 for Feature Extraction
# pooling='avg' ensures we get a 2048-dimensional vector
model = ResNet50(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))

def get_embedding(img_path):
    try:
        img = image.load_img(img_path, target_size=(224, 224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)
        return model.predict(x).flatten()
    except:
        return None

# 3. Build the Feature Index
print("Extracting features for 50 images...")
subset_df['embeddings'] = subset_df['image_path'].apply(get_embedding)
subset_df = subset_df.dropna(subset=(['embeddings']))
feature_list = np.array(subset_df['embeddings'].tolist())

# 4. Recommendation Function
def recommend_visual_products(query_img_path, top_k=5):
    query_vec = get_embedding(query_img_path).reshape(1, -1)
    
    # Calculate Cosine Similarity
    similarities = cosine_similarity(query_vec, feature_list)[0]
    indices = similarities.argsort()[-top_k:][::-1]
    
    # Plotting results
    plt.figure(figsize=(15, 5))
    plt.subplot(1, top_k + 1, 1)
    plt.imshow(image.load_img(query_img_path))
    plt.title("Query Product")
    plt.axis('off')
    
    for i, idx in enumerate(indices):
        plt.subplot(1, top_k + 1, i + 2)
        rec_path = subset_df.iloc[idx]['image_path']
        plt.imshow(image.load_img(rec_path))
        plt.title(f"Score: {similarities[idx]:.2f}")
        plt.axis('off')
    plt.show()

# 5. Execute Visual Search
# Use one of the images from the dataset as a test query
test_query = subset_df.iloc[0]['image_path']
recommend_visual_products(test_query) 